In [1]:
import sys
print(sys.executable)

import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

import torch.nn.functional as F
print("torch ok")

import torchaudio
print("torchaudio ok")

import torchcrepe
print("torchcrepe ok")

/home/ernie/miniconda3/envs/crepe/bin/python
2.11.0+cu128
12.8
True
torch ok
torchaudio ok
torchcrepe ok


In [2]:
import torchcrepe
import numpy as np
import soundfile as sf
import torchcodec

In [3]:
# Load audio
audio, sr = torchcrepe.load.audio("../audio_samples/test.wav")
print("DEBUG: prining out audio", audio)
audio = audio.mean(dim=0, keepdim=True)  # average stereo channels → mono.

# Here we'll use a 5 millisecond hop length
hop_length = int(sr / 100.)

# Provide a sensible frequency range for your domain (upper limit is 2006 Hz)
# This would be a reasonable range for speech
fmin = 50
fmax = 1550

# Select a model capacity--one of "tiny" or "full"
model = 'full'

# Choose a device to use for inference
device = 'cuda:0'

# Pick a batch size that doesn't cause memory errors on your gpu
batch_size = 2048

# Compute pitch using first gpu
pitch = torchcrepe.predict(audio,
                           sr,
                           hop_length,
                           fmin,
                           fmax,
                           model,
                           batch_size=batch_size,
                           device=device)

pitch_np = pitch.squeeze().cpu().numpy()

# reconstruct crude sine wave
samples_per_frame = hop_length
audio_out = []

phase = 0.0
for f in pitch_np:
    if f <= 0:
        chunk = np.zeros(samples_per_frame)
    else:
        t = np.arange(samples_per_frame) / sr
        chunk = np.sin(2 * np.pi * f * t + phase)
        phase += 2 * np.pi * f * samples_per_frame / sr
    audio_out.append(chunk)

audio_out = np.concatenate(audio_out)

sf.write("crepe_results/debug_pitch.wav", audio_out, sr)

DEBUG: prining out audio tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -6.1035e-05,
          3.0518e-05,  1.2207e-04],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  9.1553e-05,
          9.1553e-05,  9.1553e-05]])


/home/ernie/miniconda3/envs/crepe/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
